In [9]:
from langchain_deepseek import ChatDeepSeek
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy  # 关键导入
from langchain.tools import tool
from pydantic import BaseModel
import os

os.environ["DEEPSEEK_API_KEY"] = "sk-49892f84100b4c73843a82e371fc3522"


@tool
def search(query: str) -> str:
    """Search for information."""
    return f"Results for: {query}"


class Answer(BaseModel):
    summary: str
    confidence: float


# 关键：先创建 ChatDeepSeek 实例，把关闭 thinking 的参数传给它
model = ChatDeepSeek(
    model="deepseek-flash",
    extra_body={"thinking": {"type": "disabled"}}  # 关闭思考模式
)

agent = create_agent(
    model=model,
    tools=[search],
    system_prompt="You are a helpful assistant. Be concise and accurate.",
    response_format=ToolStrategy(Answer)  # ← 关键改动：包裹 ToolStrategy
)
result = agent.invoke({"messages": [{"role": "user", "content": "Summarize AI trends"}]})
print(result["structured_response"])  # Answer(summary=..., confidence=...)

summary='**Note:** My searches returned no actual content — only empty result headers — so the following is from general knowledge (cutoff-based), not fresh retrieval. Verify specifics before relying on them.\n\n**Major AI trends:**\n\n1. **Agentic AI** — Shift from chat assistants to autonomous agents that plan, use tools, browse, and execute multi-step workflows. Growing focus on reliability, sandboxing, and evaluation of long-horizon tasks.\n\n2. **Reasoning models** — "Test-time compute" scaling: models that think longer (chain-of-thought, search, verification) to boost accuracy on math, code, and science, trading latency/cost for capability.\n\n3. **Multimodality as default** — Text, image, audio, and video handled natively in one model; real-time voice and video interaction moving into mainstream products.\n\n4. **Smaller, cheaper, on-device models** — Distillation and quantization push capable models to phones/laptops; cost-per-token falling sharply, shifting value to inference 